### Import the required libraries

Import the PostgreSQL driver and Kafka producer, plus `json` to serialize stock messages and `time` to pause between database polls.

In [1]:
import psycopg2
from kafka import KafkaProducer
import json
import time

### Create the Kafka producer

Connect a producer to Kafka at `localhost:9092`. Configure it to serialize each message dictionary as JSON encoded into UTF-8 bytes.

In [3]:
producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

### Connect to PostgreSQL and initialize row tracking

Connect to the local `streaming_db` database and create a cursor for reading stock prices. Set `last_id = 0` so the first poll includes existing rows with positive IDs. This position is kept only in memory, so rerunning this setup can cause rows to be sent again.

In [4]:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="streaming_db",
    user="postgres",
    password=""
)
cur = conn.cursor()

last_id = 0

### Stream stock records from PostgreSQL to Kafka

Repeatedly read rows from `yahoo_finance.stock_price` whose IDs are greater than `last_id`, ordered by ID. Update the tracked ID for each row and send its symbol, date, closing price, return, and volatility to the `yahoo_finance` Kafka topic, converting the date to a string. Print each message, flush pending sends, and wait 5 seconds before polling again. The loop runs until interrupted or an unhandled error occurs.

In [5]:
while True:
    cur.execute("""
        SELECT id, symbol, date, close, return, volatility_20d
        FROM yahoo_finance.stock_price
        WHERE id > %s
        ORDER BY id
    """, (last_id,))
    
    rows = cur.fetchall()

    for r in rows:
        last_id = r[0]

        msg = {
            "symbol": r[1],
            "date": str(r[2]),
            "price": r[3],
            "return": r[4],
            "volatility": r[5]
        }

        producer.send("yahoo_finance", msg)
        print("📤 Sent:", msg)

    producer.flush()
    time.sleep(5)

📤 Sent: {'symbol': 'AAPL', 'date': '2023-02-01 00:00:00', 'price': 142.87969970703125, 'return': 0.007900876198365658, 'volatility': 0.0127579005082298}
📤 Sent: {'symbol': 'AAPL', 'date': '2023-02-02 00:00:00', 'price': 148.17520141601562, 'return': 0.03706265984490842, 'volatility': 0.014354437006802962}
📤 Sent: {'symbol': 'AAPL', 'date': '2023-02-03 00:00:00', 'price': 151.7906494140625, 'return': 0.024399818346770186, 'volatility': 0.013969202010721939}
📤 Sent: {'symbol': 'AAPL', 'date': '2023-02-06 00:00:00', 'price': 149.06922912597656, 'return': -0.017928774259752367, 'volatility': 0.013955422859601075}
📤 Sent: {'symbol': 'AAPL', 'date': '2023-02-07 00:00:00', 'price': 151.93798828125, 'return': 0.019244475684844886, 'volatility': 0.014142224733499466}
📤 Sent: {'symbol': 'AAPL', 'date': '2023-02-08 00:00:00', 'price': 149.25587463378906, 'return': -0.0176526863215809, 'volatility': 0.015311707551825059}
📤 Sent: {'symbol': 'AAPL', 'date': '2023-02-09 00:00:00', 'price': 148.224304

KeyboardInterrupt: 